In [2]:
suppressPackageStartupMessages({
    library(data.table)
    library(dplyr)
    library(gridExtra)
    library(ggpubr)
    library(ggplot2)
})

main = "/rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/"
plot_dir = paste0(main, 'results/rna/mapping/pdf')
dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)

source(paste0(main, "mapping_functions_extended.R"))


Attaching package: ‘cowplot’


The following object is masked from ‘package:ggpubr’:

    get_legend




In [3]:
meta_complete = fread(file.path(main, 'results/rna/mapping/sample_metadata_after_mapping.txt.gz'))

In [4]:
atlas_in = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/'
umap =  read.csv(paste0(atlas_in, 'umap.csv'))
atlas_meta = readRDS(paste0(atlas_in, 'integrated_meta_celltype_clus.rds'))
atlas_meta = atlas_meta[, c('cell', 'sample', 'stage', 'celltype.descendant', 'celltype.clustering')]
colnames(atlas_meta) = c('cell', 'sample', 'stage', 'celltype_original', 'celltype')
atlas_meta = merge(atlas_meta, umap, by='cell')

In [5]:
colnames(umap) = c('closest.cell_mnn', 'umapX.mapped', 'umapY.mapped')
meta_complete = merge(meta_complete, umap, by='closest.cell_mnn')

In [6]:
meta_complete$genotype = factor(ifelse(meta_complete$tdTom_corr == FALSE, 'WT', 'Stat3 KO'), levels=c('WT', 'Stat3 KO'))

In [7]:
meta_complete = as.data.table(meta_complete) %>% setnames('celltype.mapped_mnn', 'celltype.mapped')

In [8]:
head(meta_complete)

closest.cell_mnn,cell,sample,barcode,nFeature_RNA,nCount_RNA,mitochondrial_percent_RNA,ribosomal_percent_RNA,stage,tdTom,pass_rnaQC,doublet_score,doublet_call,celltype.mapped,celltype.score_mnn,idx,tdTom_corr,umapX.mapped,umapY.mapped,genotype
<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<lgl>,<lgl>,<dbl>,<lgl>,<chr>,<dbl>,<int>,<lgl>,<dbl>,<dbl>,<fct>
cell_100013,SLX-21143_SITTB2_HTJH3DSX2#TGTTACTCACTAAACC-1,SLX-21143_SITTB2_HTJH3DSX2,TGTTACTCACTAAACC-1,3772,17378,0.64,31.26,E8.5,FALSE,TRUE,0.05,FALSE,NMPs/Mesoderm-biased,0.72,50646,FALSE,11.936046,12.35801,WT
cell_100016,SLX-21143_SITTB4_HTJH3DSX2#CTACAGAAGTTTGTCG-1,SLX-21143_SITTB4_HTJH3DSX2,CTACAGAAGTTTGTCG-1,4938,22900,0.03,22.58,E9.5,FALSE,TRUE,0.21,FALSE,Dorsal hindbrain progenitors,0.92,62217,FALSE,16.588545,18.99889,WT
cell_100016,SLX-21143_SITTD4_HTJH3DSX2#TAGGGTTTCGTTTACT-1,SLX-21143_SITTD4_HTJH3DSX2,TAGGGTTTCGTTTACT-1,5019,22421,0.03,22.65,E9.5,FALSE,TRUE,0.15,FALSE,Dorsal hindbrain progenitors,0.44,84468,FALSE,16.588545,18.99889,WT
cell_100016,SLX-21143_SITTE3_HTJH3DSX2#GTAGGTTAGTAAACAC-1,SLX-21143_SITTE3_HTJH3DSX2,GTAGGTTAGTAAACAC-1,3919,16166,0.11,26.92,E9.5,TRUE,TRUE,0.17,FALSE,Dorsal hindbrain progenitors,0.84,103059,TRUE,16.588545,18.99889,Stat3 KO
cell_100017,SLX-21143_SITTG3_HTJH3DSX2#AGGACTTAGGAGAGGC-1,SLX-21143_SITTG3_HTJH3DSX2,AGGACTTAGGAGAGGC-1,4618,16999,0.04,17.70,E9.5,TRUE,TRUE,0.22,FALSE,Gut tube,0.48,140583,TRUE,2.209732,13.43087,Stat3 KO
cell_100086,SLX-21143_SITTA4_HTJH3DSX2#ACGGAAGCACGGTGAA-1,SLX-21143_SITTA4_HTJH3DSX2,ACGGAAGCACGGTGAA-1,6502,55325,0.32,27.62,E9.5,TRUE,TRUE,0.62,FALSE,Dermomyotome,0.28,8240,TRUE,16.193125,10.51507,Stat3 KO


In [10]:
p1 = ggplot(meta_complete[stage=='E7.5'], aes(umapX.mapped, umapY.mapped, col=celltype.mapped)) + 
    ggrastr::rasterize(geom_point(data=atlas_meta, aes(umapX, umapY), col='grey90'), dpi=500) + 
    #geom_point(data=atlas_meta[sample(1000),], aes(umapX, umapY), col='grey90') +
    geom_point(alpha=0.8, size=0.3) + 
    ggtitle('E7.5') + 
    facet_wrap(~genotype) + 
    scale_colour_manual(values = celltype_colours_final, name = "celltype") +
    umap_theme + 
    theme(strip.background=element_blank(),
          strip.text=element_text(size=35))

pdf(file.path(plot_dir, '/Stat3_mapping_E7.5.pdf'), width=10, height=5)
    p1
dev.off()            

png 
  2

In [11]:
legend = get_legend(ggplot(meta_complete, aes(umapX.mapped, umapY.mapped, col=celltype.mapped)) + 
    geom_point(alpha=1, size=5) + guides(col=guide_legend(ncol=3)) + 
    scale_colour_manual(values = celltype_colours_final[!is.na(unique(meta_complete[stage=='E7.5']$celltype.mapped)[match(names(celltype_colours_final), unique(meta_complete[stage=='E7.5']$celltype.mapped))])], name = "") +
    theme_void()) 
pdf(file.path(plot_dir, '/Stat3_mapping_E7.5_legend.pdf'), width=15, height=10)
    as_ggplot(legend)
dev.off()            

png 
  2

In [12]:
p1 = ggplot(meta_complete[stage=='E8.5'], aes(umapX.mapped, umapY.mapped, col=celltype.mapped)) + 
    ggrastr::rasterize(geom_point(data=atlas_meta, aes(umapX, umapY), col='grey90'), dpi=500) + 
    #geom_point(data=atlas_meta[sample(1000),], aes(umapX, umapY), col='grey90') +
    geom_point(alpha=0.8, size=0.3) + 
    ggtitle('E8.5') + 
    facet_wrap(~genotype) + 
    scale_colour_manual(values = celltype_colours_final, name = "celltype") +
    umap_theme + 
    theme(strip.background=element_blank(),
          strip.text=element_text(size=35))

pdf(file.path(plot_dir, '/Stat3_mapping_E8.5.pdf'), width=10, height=5)
    p1
dev.off()            

png 
  2

In [13]:
legend = get_legend(ggplot(meta_complete, aes(umapX.mapped, umapY.mapped, col=celltype.mapped)) + 
    geom_point(alpha=1, size=5) + guides(col=guide_legend(ncol=3)) + 
    scale_colour_manual(values = celltype_colours_final[!is.na(unique(meta_complete[stage=='E8.5']$celltype.mapped)[match(names(celltype_colours_final), unique(meta_complete[stage=='E8.5']$celltype.mapped))])], name = "") +
    theme_void()) 
pdf(file.path(plot_dir, '/Stat3_mapping_E8.5_legend.pdf'), width=15, height=10)
    as_ggplot(legend)
dev.off()            

png 
  2

In [14]:
p1 = ggplot(meta_complete[stage=='E9.5'], aes(umapX.mapped, umapY.mapped, col=celltype.mapped)) + 
    ggrastr::rasterize(geom_point(data=atlas_meta, aes(umapX, umapY), col='grey90'), dpi=500) + 
    #geom_point(data=atlas_meta[sample(1000),], aes(umapX, umapY), col='grey90') +
    geom_point(alpha=0.8, size=0.3) + 
    ggtitle('E9.5') + 
    facet_wrap(~genotype) + 
    scale_colour_manual(values = celltype_colours_final, name = "celltype") +
    umap_theme + 
    theme(strip.background=element_blank(),
          strip.text=element_text(size=35))

pdf(file.path(plot_dir, '/Stat3_mapping_E9.5.pdf'), width=10, height=5)
    p1
dev.off()            

png 
  2

In [15]:
legend = get_legend(ggplot(meta_complete, aes(umapX.mapped, umapY.mapped, col=celltype.mapped)) + 
    geom_point(alpha=1, size=5) + guides(col=guide_legend(ncol=3)) + 
    scale_colour_manual(values = celltype_colours_final[!is.na(unique(meta_complete[stage=='E9.5']$celltype.mapped)[match(names(celltype_colours_final), unique(meta_complete[stage=='E9.5']$celltype.mapped))])], name = "") +
    theme_void()) 
pdf(file.path(plot_dir, '/Stat3_mapping_E9.5_legend.pdf'), width=15, height=10)
    as_ggplot(legend)
dev.off()            

png 
  2